# Finding datasets fit for TSMD using Clustering
I aim to find classification datasets that I can use to create TSMD
benchmarks. A classification dataset is fit for creating TSMD time
series if the instances belonging to different classes are distinguishable in an
unsupervised manner. I assess this by applying a centroid-based clustering
algorithm (k-medoids) and observing whether a good clustering can be achieved.
I use TimeSeriesClassification (TSC) datasets with at least 5 classes (c ≥ 5).
The datasets can be categorized based on whether they are fixed-length or
variable-length and whether they are univariate or multivariate.
For the first category (fixed-length, univariate), I can use the clustering benchmark paper. I cluster
the datasets from the remaining three categories myself.

All datasets are clustered with k-medoids because it can for both Euclidean
distance (ED) and Dynamic Time Warping (DTW) distance (in contrast to k-means). All instances are z-normalized beforehand and I use use k = c. To apply k-medoids with Euclidean distance on a variable-length dataset, I resample instances from the dataset such that their lengths are equal to the average length in the dataset. DTW is directly applicable. To apply k-medoids with DTW on a multivariate dataset, I use dependent DTW.

In [1]:
import os
PATH_TO_DATA = os.path.join("..", "ucr_datasets")
PATH_TO_RESULTS = os.path.join(".", "cluster_results")

In [2]:
import numpy as np

import kmedoids
from sklearn.metrics import adjusted_rand_score

import matplotlib.pyplot as plt
import pickle
import pandas as pd

import sys
sys.path.insert(0, "..") 
import read_ucr

In [3]:
import dtaidistance.dtw_ndim as dtw

def ed(s1, s2):
    return dtw.distance(s1, s2, window=1, use_c=True)

def dtwdistance(s1, s2, **dtw_params):
    return dtw.distance(s1, s2, use_c=True, **dtw_params)

In [4]:
from tslearn.preprocessing import TimeSeriesResampler

def resample_series(series, length):
    resampled = TimeSeriesResampler(sz=length).transform(series.T).T
    return resampled[0]

def znormalize(series):
    series = (series - np.mean(series, axis=None)) / np.std(series, axis=None)
    return series

/cw/dtaijupiter/NoCsBack/dtai/daanv/miniconda3/envs/md/lib/python3.10/site-packages/tslearn/bases/bases.py:15: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


In [5]:
# Datasets with more than 5 classes
VAR_LENGTH_UNIVARIATE   = ["AllGestureWiimoteX", "AllGestureWiimoteY", "AllGestureWiimoteZ", "GestureMidAirD1", "GestureMidAirD2", "GestureMidAirD3", "GesturePebbleZ1", "ShakeGestureWiimoteZ",  "GesturePebbleZ2", "PickupGestureWiimoteZ", "PLAID"]
VAR_LENGTH_MULTIVARIATE = ['CharacterTrajectories', 'JapaneseVowels', 'SpokenArabicDigits']
FIX_LENGTH_MULTIVARIATE = ["ArticularyWordRecognition", "Cricket", "DuckDuckGeese", "EigenWorms", "Ering", "Handwriting", "Libras", "LSST", "NATOPS", "PenDigits", "PEMS-SF", "PhonemeSpectra", "UWaveGestureLibrary"]
DATASETS = VAR_LENGTH_UNIVARIATE + VAR_LENGTH_MULTIVARIATE + FIX_LENGTH_MULTIVARIATE

In [6]:
from functools import partial

for ds_name in DATASETS:
    path_to_dataset = os.path.join(PATH_TO_DATA, ds_name.lower())
    path_to_file = os.path.join(path_to_dataset, "znormalized.pkl")
    
    if os.path.exists(path_to_file):
        continue
    
    df_train, df_test = read_ucr.load_ucr_dataset(PATH_TO_DATA, ds_name, train=True), read_ucr.load_ucr_dataset(PATH_TO_DATA, ds_name, train=False)
    df = pd.concat((df_train, df_test), axis=0)
    df.to_pickle(os.path.join(path_to_dataset, "znormalized.pkl"))

In [7]:
from functools import partial
import parallel_cdist as pp

for ds_name in DATASETS:
    print(ds_name)
    # Load the dataset
    path_to_file = os.path.join(PATH_TO_DATA, ds_name.lower(), "znormalized.pkl")
    df = pd.read_pickle(path_to_file)
    
    # Resample the dataset
    l_mean = int(df['length'].mean())
    df['resampled'] = df['series'].apply(lambda instance: resample_series(instance, l_mean))
    
    # Number of classes
    nclasses = len(np.unique(df['label']))
    
    # Make a dir
    path_to_result = os.path.join(PATH_TO_RESULTS, ds_name.lower())
    if not os.path.exists(path_to_result):
        os.mkdir(path_to_result)
    
    # Cluster using DTW
    path_to_clustering = os.path.join(path_to_result, f"clustering_dtw.pkl")
    if not os.path.exists(path_to_clustering):
        distance_partial = partial(dtwdistance, **{})
        D = pp.cdist_generic(distance_partial, dataset1=df['series'].tolist(), n_jobs=4, progress_bar=True, parallel_method='processes')
        np.random.seed(0)
        cluster_result = kmedoids.fastpam1(D, nclasses)
        with open(path_to_clustering, "wb") as ofile:
            pickle.dump(cluster_result, ofile)
    
    # Cluster using ED
    path_to_clustering = os.path.join(path_to_result, f"clustering_ed.pkl")
    if not os.path.exists(path_to_clustering):
        distance_partial = partial(ed, **{})
        D = pp.cdist_generic(distance_partial, dataset1=df['resampled'].tolist(), n_jobs=4, progress_bar=True, parallel_method='processes')    
        np.random.seed(0)
        cluster_result = kmedoids.fastpam1(D, nclasses)
        with open(path_to_clustering, "wb") as ofile:
            pickle.dump(cluster_result, ofile)

AllGestureWiimoteX
AllGestureWiimoteY
AllGestureWiimoteZ
GestureMidAirD1
GestureMidAirD2
GestureMidAirD3
GesturePebbleZ1
ShakeGestureWiimoteZ
GesturePebbleZ2
PickupGestureWiimoteZ
PLAID
CharacterTrajectories
JapaneseVowels
SpokenArabicDigits
ArticularyWordRecognition
Cricket
DuckDuckGeese
EigenWorms
Ering
Handwriting
Libras
LSST
NATOPS
PenDigits
PEMS-SF
PhonemeSpectra
UWaveGestureLibrary


In [8]:
def plot_clustering(dataset, cluster_labels):
    nclasses = len(np.unique(cluster_labels))
    ncols = 3
    nrows = int(np.ceil(nclasses / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axs = axs.flatten()
    
    N = len(dataset)
    for i in range(N):
        axs[cluster_labels[i]].set_prop_cycle(None)
        axs[cluster_labels[i]].plot(dataset[i], alpha=0.5)
    
    return fig, axs

In [ ]:
df_results = pd.DataFrame(columns=['dataset', 'distance_measure', 'ari'])

for ds_name in DATASETS:
    print(ds_name)
    
    # Load the dataset
    path_to_file = os.path.join(PATH_TO_DATA, ds_name.lower(), "znormalized.pkl")
    df = pd.read_pickle(path_to_file)
    y  = df['label']

    # Resample the dataset
    l_mean = int(df['length'].mean())
    df['resampled'] = df['series'].apply(lambda instance: resample_series(instance, l_mean))
    
    # Load ED Result
    instances = df['resampled'].tolist()
    path_to_result = os.path.join(PATH_TO_RESULTS, ds_name.lower())
    path_to_clustering = os.path.join(path_to_result, f"clustering_ed.pkl")
    with open(path_to_clustering, "rb") as ifile:
        cluster_result = pickle.load(ifile)

    # Calculate ARI
    cluster_labels    = cluster_result.labels
    ari = adjusted_rand_score(y, cluster_labels)
    
    # Store ED Result
    new_row = {'dataset':ds_name, 'distance_measure': 'ed', 'ari': ari}
    df_results = pd.concat((df_results, pd.DataFrame([new_row])), ignore_index=True)
    
    # Plot the ED clustering
    fig, ax = plot_clustering(instances, cluster_labels)
    fig.suptitle(f'{ds_name} using ED:  ARI = {ari:.2f}')
    plt.savefig(os.path.join(path_to_result, "clustering_ed.png"))
    plt.close(fig)
        
    # Load DTW Result
    instances = df['series'].tolist()
    path_to_result = os.path.join(PATH_TO_RESULTS, ds_name.lower())
    path_to_clustering = os.path.join(path_to_result, f"clustering_dtw.pkl")
    with open(path_to_clustering, "rb") as ifile:
        cluster_result = pickle.load(ifile)

    # Calculate ARI
    cluster_labels    = cluster_result.labels
    ari = adjusted_rand_score(y, cluster_labels)
    
    # Store DTW Result
    new_row = {'dataset':ds_name, 'distance_measure': 'dtw', 'ari': ari}
    df_results = pd.concat((df_results, pd.DataFrame([new_row])), ignore_index=True)
 
    # Plot the clustering
    fig, ax = plot_clustering(instances, cluster_labels)
    fig.suptitle(f'{ds_name} using DTW: ARI = {ari:.2f}')
    plt.savefig(os.path.join(path_to_result, "clustering_dtw.png"))
    plt.close(fig)

AllGestureWiimoteX
AllGestureWiimoteY
AllGestureWiimoteZ
GestureMidAirD1
GestureMidAirD2
GestureMidAirD3
GesturePebbleZ1
ShakeGestureWiimoteZ
GesturePebbleZ2
PickupGestureWiimoteZ
PLAID
CharacterTrajectories
JapaneseVowels
SpokenArabicDigits
ArticularyWordRecognition
Cricket
DuckDuckGeese
EigenWorms
Ering
Handwriting
Libras
LSST
NATOPS
PenDigits
PEMS-SF


In [ ]:
df_results